# MINT layer sweep for the contact head

Which MINT layer works best for inter-chain contact prediction? This notebook trains the same
`contact/train_contact_head.py` head once per layer and compares them on a held-out validation split.

1. **Split.** Everything comes from one labelled file, `trainset_contacts.json` (made by
   `contact/label_contacts.py`). `N_VAL` clusters are held out for validation and `N_TRAIN` of the rest are used for training.
2. Extract embeddings for both splits.
3. Train one head per layer (`--layers L`), plus optionally one on all of them with the learned layer mix.
4. Compare best validation AP / top-L/5 precision.

**Runtime → Change runtime type → GPU** (T4 is enough).

`data/` is gitignored, so upload `interaction_head/data/trainset_contacts.json` to `CONTACTS` on Drive first.
The branch in `BRANCH` must contain `contact/train_contact_head.py` and `extract_residue_embeddings.py`, so push them first.

Re-running is safe after a disconnect. Splits, embeddings and finished runs are saved on Drive and
reused, so each step only ever runs once. Each session copies the embeddings to the Colab disk before
training, because reading them straight from Drive is slow.

In [ ]:
# ---- settings ----
DRIVE_DIR  = "/content/drive/MyDrive/mint"
CKPT       = f"{DRIVE_DIR}/mint.ckpt"               # cached so it's only downloaded once (~3.2 GB)
CONTACTS   = f"{DRIVE_DIR}/trainset_contacts.json"  # label_contacts.py output; upload interaction_head/data/trainset_contacts.json here
SWEEP_DIR  = f"{DRIVE_DIR}/layer_sweep"             # splits, runs, results
EMB_DIR    = f"{SWEEP_DIR}/emb"                     # embeddings extracted here are saved and reused
LOCAL_EMB  = "/content/emb"                         # training reads a local copy (much faster than Drive); None = read Drive directly

REPO_URL   = "https://github.com/AlecSilver/mint.git"
BRANCH     = "interaction_head"

# data
SWEEP_LAYERS = [24, 30, 33]   # 33 = final layer
MIX_RUN      = True    # also train one head on all SWEEP_LAYERS with the learned layer mix
N_VAL        = 500     # clusters held out of CONTACTS for validation
N_TRAIN      = 2000    # train clusters from the rest; None = all of them (embeddings for every pair take a lot of Drive space)
DATA_SEED    = 0       # which clusters go where
CUTOFF       = 8.0     # Å, contact definition; must match the one CONTACTS was labelled with
MAX_TOKENS   = 16384   # extraction token budget per forward pass; lower it on CUDA out-of-memory
CROP         = "interface"  # extraction window for chains > 510 residues

# head training: identical for every run, so only the layers differ
PAIR       = "add"     # add | multiply
HIDDEN     = 128
EPOCHS     = 10
BATCH_SIZE = 4
TRAIN_CROP = 256
ALL_PAIRS  = True      # every train pair each epoch; False = one per cluster (same thing when CONTACTS has one pair per cluster)
TRAIN_SEED = 0         # change it and re-run to see how much of a gap between layers is noise
TRAIN_ARGS = ""        # anything else for train_contact_head.py, e.g. "--dist-weight 0.5"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(SWEEP_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Code and checkpoint

If the repo is private, use a URL with a personal access token
(`https://<token>@github.com/AlecSilver/mint.git`).

In [ ]:
REPO_DIR = "/content/mint"
if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
HEAD_DIR = f"{REPO_DIR}/interaction_head"
for f in ["extract_residue_embeddings.py", "contact/train_contact_head.py"]:
    assert os.path.exists(f"{HEAD_DIR}/{f}"), f"{f} is missing on branch {BRANCH}; push it first"
assert os.path.exists(CONTACTS), f"{CONTACTS} not found; upload interaction_head/data/trainset_contacts.json there"

# Colab already has torch, numpy, scipy, pandas and tqdm
!pip install -q --no-deps -e {REPO_DIR}

In [ ]:
if not os.path.exists(CKPT):
    !wget -q --show-progress -O {CKPT}.part https://huggingface.co/varunullanat2012/mint/resolve/main/mint.ckpt
    os.replace(CKPT + ".part", CKPT)

## 1. Split

Clusters are shuffled with `DATA_SEED`. The first `N_VAL` are validation and the next `N_TRAIN` are
train, so the two never share a cluster. The validation split depends only on `N_VAL` and the seed, so
changing `N_TRAIN` keeps the same validation pairs (and reuses their embeddings).

File names include the sizes and seed, so changing them makes new files instead of reusing stale ones.

In [ ]:
import glob, json, random

STEM = os.path.splitext(os.path.basename(CONTACTS))[0]
TAGS = {"val": f"{STEM}_v{N_VAL}_s{DATA_SEED}",
        "train": f"{STEM}_n{N_TRAIN or 'all'}_v{N_VAL}_s{DATA_SEED}"}
SPLITS = ["train", "val"]
SPLIT_FILE = {s: f"{SWEEP_DIR}/{s}_{TAGS[s]}_contacts.json" for s in SPLITS}

if all(os.path.exists(p) for p in SPLIT_FILE.values()):
    print("exists:", *SPLIT_FILE.values())
else:
    by_cluster = {}
    with open(CONTACTS) as f:
        for line in f:
            rec = json.loads(line)
            assert rec["cutoff"] == CUTOFF, f"{CONTACTS} was labelled at {rec['cutoff']} Å, not CUTOFF = {CUTOFF}"
            by_cluster.setdefault(rec["cluster"], []).append(line)
    clusters = sorted(by_cluster)
    random.Random(DATA_SEED).shuffle(clusters)
    picked = {"val": clusters[:N_VAL],
              "train": clusters[N_VAL:N_VAL + N_TRAIN] if N_TRAIN else clusters[N_VAL:]}
    for split in SPLITS:
        dst = SPLIT_FILE[split]
        with open(dst + ".part", "w") as f:
            for c in picked[split]:
                f.writelines(by_cluster[c])
        os.replace(dst + ".part", dst)
        n = sum(len(by_cluster[c]) for c in picked[split])
        print(f"{split}: {n} pairs in {len(picked[split])} of {len(clusters)} clusters -> {dst}")

## 2. Embeddings

Extracts both splits to Drive. A split is skipped once it's complete, and resumes from the last
finished shard if it was cut off. The folder name includes the layers and crop mode, so changing either extracts a new set.

In [ ]:
LAYERS_STR = " ".join(map(str, sorted(SWEEP_LAYERS)))
%cd {HEAD_DIR}
EMB = {}  # split -> embedding folder on Drive
for split in SPLITS:
    EMB[split] = out = f"{EMB_DIR}/{split}_{TAGS[split]}_L{'-'.join(map(str, sorted(SWEEP_LAYERS)))}_{CROP}"
    if os.path.exists(f"{out}/DONE"):  # skip loading the 3.2 GB checkpoint just to find nothing to do
        print("exists:", out)
        continue
    !python extract_residue_embeddings.py {SPLIT_FILE[split]} {out} --checkpoint {CKPT} --max-tokens {MAX_TOKENS} --crop {CROP} --layers {LAYERS_STR}
    assert _exit_code == 0, f"extraction failed for {split}"
    open(f"{out}/DONE", "w").close()

# local copy for training; rsync only copies what's missing, so re-running is cheap
EMB_READ = dict(EMB)
if LOCAL_EMB:
    for split in SPLITS:
        !mkdir -p {LOCAL_EMB} && rsync -a --info=progress2 {EMB[split]} {LOCAL_EMB}/
        assert _exit_code == 0, f"copying {split} embeddings failed (disk full?)"
        EMB_READ[split] = f"{LOCAL_EMB}/{os.path.basename(EMB[split])}"
TRAIN_EMB, VAL_EMB = EMB_READ["train"], EMB_READ["val"]
!du -sh {TRAIN_EMB} {VAL_EMB}

## 3. Train one head per layer

Each run writes `best.pt`, `last.pt` and `log.jsonl` to Drive. A finished run is skipped when you
re-run this cell. An unfinished one (cut off by a disconnect) is restarted from scratch.
The run folder name includes a hash of the training settings, so changing them starts a new sweep.

In [ ]:
import hashlib, shutil

settings = dict(pair=PAIR, hidden=HIDDEN, epochs=EPOCHS, batch=BATCH_SIZE, crop=TRAIN_CROP,
                all_pairs=ALL_PAIRS, seed=TRAIN_SEED, cutoff=CUTOFF, extra=TRAIN_ARGS)
RUN_ROOT = f"{SWEEP_DIR}/runs/{TAGS['train']}_{PAIR}_h{HIDDEN}_e{EPOCHS}_" + \
           hashlib.md5(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:6]
os.makedirs(RUN_ROOT, exist_ok=True)
with open(f"{RUN_ROOT}/settings.json", "w") as f:
    json.dump(settings, f, indent=2)

RUNS = {f"L{l}": [l] for l in sorted(SWEEP_LAYERS)}
if MIX_RUN and len(SWEEP_LAYERS) > 1:
    RUNS["mix"] = sorted(SWEEP_LAYERS)
ALL_PAIRS_ARG = "--all-pairs" if ALL_PAIRS else ""

def finished(out):
    log = f"{out}/log.jsonl"
    return os.path.exists(log) and sum(1 for _ in open(log)) >= EPOCHS

%cd {HEAD_DIR}
for name, layers in RUNS.items():
    out = f"{RUN_ROOT}/{name}"
    if finished(out):
        print(f"{name}: done, skipping")
        continue
    shutil.rmtree(out, ignore_errors=True)  # log.jsonl appends, so never continue a partial run
    layers_str = " ".join(map(str, layers))
    print(f"==== {name}: layers {layers_str}")
    !python contact/train_contact_head.py --train-emb {TRAIN_EMB} --train-contacts {SPLIT_FILE['train']} --val-emb {VAL_EMB} --val-contacts {SPLIT_FILE['val']} --out {out} --layers {layers_str} --pair {PAIR} --hidden {HIDDEN} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --crop {TRAIN_CROP} --cutoff {CUTOFF} --seed {TRAIN_SEED} {ALL_PAIRS_ARG} {TRAIN_ARGS}
    assert _exit_code == 0, f"training failed for {name}"

## 4. Compare

Each run is scored at its best epoch by validation AP. AP is computed over all validation cells.
Top-L/5 is the share of each pair's L/5 highest-scoring cells that are true contacts (L = shorter chain).

In [ ]:
import pandas as pd

rows = []
for name, layers in RUNS.items():
    log = f"{RUN_ROOT}/{name}/log.jsonl"
    if not os.path.exists(log):
        continue
    epochs = [json.loads(l) for l in open(log)]
    best = max(epochs, key=lambda r: r["val_ap"])
    row = {"run": name, "layers": layers, "epochs_done": len(epochs), "best_epoch": best["epoch"],
           "val_ap": best["val_ap"], "top10": best["val_top10"], "topL/10": best["val_topL/10"],
           "topL/5": best["val_topL/5"], "last_ap": epochs[-1]["val_ap"]}
    if len(layers) > 1:
        row["layer_weights"] = {l: round(w, 3) for l, w in zip(layers, best["layer_weights"])}
    rows.append(row)

results = pd.DataFrame(rows)
results.to_csv(f"{RUN_ROOT}/results.csv", index=False)
print("positive rate (AP of a random guess):", round(epochs[-1]["val_pos_rate"], 4))
results.round(4)

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID, SERIES = "#1f2328", "#59636e", "#d8dee4", "#2f6fdf"
single = results[results.layers.map(len) == 1].reset_index(drop=True)  # already in layer order
x = [l[0] for l in single.layers]
mix = results[results.run == "mix"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
for ax, col, title in zip(axes, ["val_ap", "topL/5"], ["Validation AP", "Validation top-L/5 precision"]):
    ax.plot(x, single[col], color=SERIES, lw=2, marker="o", ms=7, mec="white", mew=1.5, zorder=3)
    b = single[col].idxmax()
    ax.annotate(f"{single[col][b]:.3f}", (single.layers[b][0], single[col][b]), textcoords="offset points",
                xytext=(0, 9), ha="center", color=INK, fontsize=9)
    if len(mix):
        y = mix[col].iloc[0]
        ax.axhline(y, color=MUTED, lw=1.2, ls="--", zorder=2)
        ax.annotate(f"layer mix {y:.3f}", (x[-1], y), textcoords="offset points", xytext=(0, -13),
                    ha="right", color=MUTED, fontsize=9)
    ax.margins(y=0.25)
    ax.set_title(title, loc="left", color=INK, fontsize=11, pad=12)
    ax.set_xlabel("MINT layer", color=MUTED)
    ax.set_xticks(x)
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.tick_params(colors=MUTED, length=0)
    for s in ax.spines.values():
        s.set_visible(False)
fig.tight_layout()
fig.savefig(f"{RUN_ROOT}/layer_sweep.png", dpi=150)
plt.show()
print("saved to", RUN_ROOT)